# DAC Latent Inspector
Load and inspect `.pt` files containing DAC (Descript Audio Codec) latent representations.

**Usage**: Set a file path or select by index to view latent features, shapes, metadata, and visualizations.

## 2) Import Required Libraries

In [1]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# If you want nicer tables, optionally install pandas and uncomment below:
# import pandas as pd

## 3) Load DAC Latent Files
Set the base directory or provide a direct path to a `.pt` file.

In [2]:
# Change this to where your .pt files live
base_dir = Path("../data").resolve()

pt_files = sorted(base_dir.rglob("*.pt"))
print(f"Found {len(pt_files)} .pt files under {base_dir}")

Found 2112 .pt files under C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data


In [3]:
# List files with indices for easy selection
for i, p in enumerate(pt_files[:50]):
    print(f"{i:03d}: {p}")
if len(pt_files) > 50:
    print(f"... and {len(pt_files) - 50} more")

000: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\00000_Preludes Book 2 -.pt
001: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00000_instrum_mel.pt
002: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00001_instrum_mel.pt
003: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00002_instrum_mel.pt
004: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00003_instrum_mel.pt
005: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00004_instrum_mel.pt
006: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00005_instrum_mel.pt
007: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00006_instrum_mel.pt
008: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00007_instrum_mel.pt
009: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00008_instrum_mel.pt
010: C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00

## 4) Load a Specific DAC Latent File
Either select by index from the list above, or provide a direct path.

In [ ]:
# Option 1: Pick by index from the list above
index = 0

# Option 2: Or set a direct path (uncomment and edit)
# pt_path = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\00000_Preludes Book 2 -.pt")

if len(pt_files) == 0:
    raise FileNotFoundError("No .pt files found. Check base_dir.")

if 'pt_path' not in locals():
    pt_path = pt_files[index]

data = torch.load(pt_path, map_location="cpu")
print(f"✓ Loaded: {pt_path.name}")
print(f"File size: {pt_path.stat().st_size / 1024:.1f} KB")
print(f"Type: {type(data)}")
print(f"Keys: {list(data.keys()) if isinstance(data, dict) else 'Not a dict'}")

## 5) View DAC Latent Features
Show metadata and tensor shapes for the loaded latent.

In [ ]:
if not isinstance(data, dict):
    print("⚠ Not a dictionary. Cannot inspect DAC latent structure.")
else:
    print("\n=== DAC Latent Contents ===")
    
    # Show metadata
    if 'sample_rate' in data:
        print(f"Sample Rate: {data['sample_rate']} Hz")
    if 'original_length' in data:
        print(f"Original Length: {data['original_length']} samples")
        if 'sample_rate' in data:
            duration = data['original_length'] / data['sample_rate']
            print(f"Duration: {duration:.2f} seconds")
    
    print("\n=== Tensor Features ===")
    for key in ['z', 'codes', 'latents']:
        if key in data and torch.is_tensor(data[key]):
            t = data[key]
            print(f"\n{key}:")
            print(f"  Shape: {tuple(t.shape)}")
            print(f"  Dtype: {t.dtype}")
            print(f"  Device: {t.device}")
            if t.numel() > 0:
                t_np = t.detach().cpu().numpy()
                print(f"  Range: [{t_np.min():.4f}, {t_np.max():.4f}]")
                print(f"  Mean: {t_np.mean():.4f}, Std: {t_np.std():.4f}")

## 6) Visualize Latent Codes
Plot the discrete codebook indices over time (if available).

In [ ]:
if 'codes' in data and torch.is_tensor(data['codes']):
    codes = data['codes'].cpu().numpy()
    # codes shape: (batch, num_codebooks, time)
    
    if codes.ndim == 3:
        batch_idx = 0
        codes_2d = codes[batch_idx]  # (num_codebooks, time)
        
        fig, ax = plt.subplots(figsize=(12, 4))
        im = ax.imshow(codes_2d, aspect='auto', cmap='viridis', interpolation='nearest')
        ax.set_xlabel('Time Step')
        ax.set_ylabel('Codebook Index')
        ax.set_title(f'DAC Codes - {pt_path.name}')
        plt.colorbar(im, ax=ax, label='Code Index')
        plt.tight_layout()
        plt.show()
        
        print(f"Codes shape: {codes_2d.shape}")
        print(f"Number of codebooks: {codes_2d.shape[0]}")
        print(f"Time steps: {codes_2d.shape[1]}")
    else:
        print(f"Unexpected codes shape: {codes.shape}")
else:
    print("No 'codes' tensor found.")

## 7) Visualize Latent Distribution
Plot histogram of quantized latent values ('z').

In [ ]:
if 'z' in data and torch.is_tensor(data['z']):
    z = data['z'].cpu().numpy()
    z_flat = z.ravel()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    ax1.hist(z_flat, bins=80, edgecolor='black', alpha=0.7)
    ax1.set_xlabel('Latent Value')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribution of Quantized Latents (z)')
    ax1.grid(alpha=0.3)
    
    # Time series of first few latent dimensions
    if z.ndim == 3:  # (batch, latent_dim, time)
        batch_idx = 0
        z_2d = z[batch_idx]  # (latent_dim, time)
        num_dims_to_plot = min(5, z_2d.shape[0])
        for i in range(num_dims_to_plot):
            ax2.plot(z_2d[i, :500], alpha=0.7, label=f'Dim {i}')
        ax2.set_xlabel('Time Step')
        ax2.set_ylabel('Latent Value')
        ax2.set_title(f'First {num_dims_to_plot} Latent Dimensions (first 500 steps)')
        ax2.legend()
        ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No 'z' tensor found.")